### Disney+Hulu Regional Data EDA

In [2]:
import pandas as pd
import numpy as np

pd.set_option("mode.copy_on_write", True)

In [3]:
disney_regional = pd.read_csv("data/disney_regional_panel.csv")

disney_regional.head()

,quarter,region_proxy,platform,subscribers_m,arpu_usd,Revenue (Millions)
0,2022Q3,UCAN,Disney+,44.5,6.10,271.450
1,2022Q3,EMEA_INTL,Disney+,61.4,5.83,357.962
2,2022Q3,APAC_HOTSTAR,Disney+,58.4,0.58,33.872
3,2022Q4,UCAN,Disney+,46.6,5.95,277.270
4,2022Q4,EMEA_INTL,Disney+,56.1,5.95,333.795


In [4]:
disney_regional.columns

Index(['quarter', 'region_proxy', 'platform', 'subscribers_m', 'arpu_usd',
       'Revenue (Millions)'],
      dtype='object')

In [5]:
disney_regional.info()
disney_regional.describe(include="all")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   quarter             21 non-null     object 
 1   region_proxy        21 non-null     object 
 2   platform            21 non-null     object 
 3   subscribers_m       21 non-null     float64
 4   arpu_usd            21 non-null     float64
 5   Revenue (Millions)  21 non-null     float64
dtypes: float64(3), object(3)
memory usage: 1.1+ KB


,quarter,region_proxy,platform,subscribers_m,arpu_usd,Revenue (Millions)
count,21,21,21,21.000000,21.000000,21.0000
unique,7,3,1,NaN,NaN,NaN
top,2022Q3,UCAN,Disney+,NaN,NaN,NaN
freq,3,7,21,NaN,NaN,NaN
mean,NaN,NaN,NaN,52.095238,4.709524,252.5790
std,NaN,NaN,NaN,8.536128,2.931371,163.7957
min,NaN,NaN,NaN,36.000000,0.580000,25.2000
25%,NaN,NaN,NaN,46.300000,1.050000,39.4800
50%,NaN,NaN,NaN,54.700000,5.950000,331.2000
75%,NaN,NaN,NaN,58.400000,6.840000,361.5620


In [ ]:
df = disney_regional.copy()

df.columns = df.columns.str.lower().str.strip()

# rename columns for clarity
df = df.rename(columns={
    "region_proxy": "region",
    "subscribers_m": "subscribers",
    "revenue (millions)": "revenue"
})

In [12]:
df["year"] = df["quarter"].str[:4].astype(int)
df["q"] = df["quarter"].str[-1].astype(int)

# Convert to actual dates (end of quarter)
df["date"] = pd.PeriodIndex(df["quarter"], freq="Q").to_timestamp(how="end")
df["date"] = df["date"].dt.normalize()

In [13]:
df[["quarter", "date"]].head()

,quarter,date
0,2022Q3,2022-09-30
1,2022Q3,2022-09-30
2,2022Q3,2022-09-30
3,2022Q4,2022-12-31
4,2022Q4,2022-12-31


In [14]:
# Create Net Ads 
df = df.sort_values(["region", "date"])

df["net_adds"] = df.groupby("region")["subscribers"].diff()

In [15]:
# Drop first rows
df = df.dropna(subset=["net_adds"])

In [19]:
df.isna().sum()

quarter        0
region         0
platform       0
subscribers    0
arpu_usd       0
revenue        0
year           0
q              0
date           0
net_adds       0
is_netflix     0
dtype: int64

In [16]:
# Add platform indicator
df["is_netflix"] = 0

In [18]:
df.head()

,quarter,region,platform,subscribers,arpu_usd,revenue,year,q,date,net_adds,is_netflix
5,2022Q4,APAC_HOTSTAR,Disney+,57.5,0.59,33.925,2022,4,2022-12-31,-0.9,0
8,2023Q1,APAC_HOTSTAR,Disney+,52.9,0.59,31.211,2023,1,2023-03-31,-4.6,0
11,2023Q2,APAC_HOTSTAR,Disney+,52.9,0.59,31.211,2023,2,2023-06-30,0.0,0
14,2023Q3,APAC_HOTSTAR,Disney+,37.6,1.05,39.480,2023,3,2023-09-30,-15.3,0
17,2023Q4,APAC_HOTSTAR,Disney+,38.3,1.28,49.024,2023,4,2023-12-31,0.7,0


In [ ]:
df.groupby("region").size()

region
APAC_HOTSTAR    6
EMEA_INTL       6
UCAN            6
dtype: int64

In [20]:
#Save cleaned data
df.to_csv("data/cleaned/disney_regional_cleaned.csv", index=False)

In [21]:
# unique values in region
df["region"].unique()

array(['APAC_HOTSTAR', 'EMEA_INTL', 'UCAN'], dtype=object)